# ATML PA3 Task 1
Run one phase per Kaggle session. Enable GPU and Internet in Notebook Settings. Add this repository as a Kaggle input dataset, or clone it into `/kaggle/working`. Keep `/kaggle/working` outputs as a Kaggle dataset between sessions. The assignment prompt YAML must be present at the repository root for the qualitative phase.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/kaggle/input/atml-pa3')  # edit if your input dataset has another path
if not (REPO / 'task1' / 'run.py').exists():
    REPO = Path('/kaggle/working/ATML-PA3')
if not (REPO / 'task1' / 'run.py').exists():
    raise FileNotFoundError('Add the repository as a Kaggle input dataset or clone it to /kaggle/working/ATML-PA3')
os.chdir(REPO)
print('Repository:', REPO)

## Restore a previous session
Attach your saved output dataset in Kaggle's Add Input panel. Set its mounted path below and run this cell before any phase.

In [ ]:
import shutil
RESTORE_DATASETS = []  # e.g. [Path('/kaggle/input/atml-pa3-results')]
RESTORE_MODE = 'train'  # 'train' or 'analysis'
RESTORE_METHOD = 'full'  # method whose latest.pt is needed for training
for dataset in map(Path, RESTORE_DATASETS):
    source_root = dataset / 'outputs/task1'
    if not source_root.exists():
        raise FileNotFoundError(source_root)
    target_root = Path('/kaggle/working/outputs/task1')
    target_root.mkdir(parents=True, exist_ok=True)
    if (source_root / 'data').exists():
        shutil.copytree(source_root / 'data', target_root / 'data', dirs_exist_ok=True)
    result_source = dataset / 'results/task1'
    if result_source.exists():
        shutil.copytree(result_source, Path('/kaggle/working/results/task1'), dirs_exist_ok=True)
    for method_dir in source_root.iterdir():
        if not method_dir.is_dir() or method_dir.name == 'data':
            continue
        if RESTORE_MODE == 'train' and method_dir.name != RESTORE_METHOD:
            continue
        target = target_root / method_dir.name
        target.mkdir(parents=True, exist_ok=True)
        for file in method_dir.iterdir():
            if not file.is_file() or file.name.startswith('recovery_'):
                continue  # prior recovery files remain preserved in the attached input dataset
            if RESTORE_MODE == 'analysis' and file.name == 'latest.pt':
                continue
            shutil.copy2(file, target / file.name)
    print('Restored selected files from', dataset)

## Installation and GPU diagnostics
Run this after a fresh Kaggle session starts. Restart the kernel if pip replaces an imported dependency.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'task1.run', 'diagnose'], check=True)

## Select one phase
Set `PHASE`, `METHOD`, and `CONDITION` below. Each train phase resumes from its latest valid recovery checkpoint. Copy saved outputs back to `/kaggle/working/outputs/task1` before continuing in a new session.

In [ ]:
PHASE = 'prepare'  # prepare, evaluate, train, weight, activations, qualitative, aggregate, export
METHOD = 'full'  # full, l2sp, lora4, lora16, oplora16
CONDITION = 'baseline'  # baseline, full_mid, full_final, l2sp_final, lora4_final, lora16_final, oplora16_final
STOP_UPDATE = None  # optionally 312, 625, 937, or 1250 for staged training
cmd = [sys.executable, '-m', 'task1.run', PHASE]
if PHASE in ('train', 'export'):
    cmd += ['--method', METHOD]
if PHASE == 'train' and STOP_UPDATE is not None:
    cmd += ['--stop-update', str(STOP_UPDATE)]
if PHASE in ('evaluate', 'qualitative'):
    cmd += ['--condition', CONDITION]
print('Running:', ' '.join(cmd), flush=True)
subprocess.run(cmd, cwd=REPO, check=True)

## Order of phases
1. `prepare` then `evaluate` with `baseline`.
2. `train` separately for `full`, `l2sp`, `lora4`, `lora16`, `oplora16`. Set `STOP_UPDATE` to a required checkpoint if the disk estimate prevents a single-session run.
3. `evaluate` for the remaining six conditions.
4. `weight`, `activations`, and `qualitative` for each condition.
5. `aggregate`, then `export` each training method.

Kaggle session storage is temporary. Save `/kaggle/working/outputs/task1` and `/kaggle/working/results/task1` as a Kaggle dataset before session end. Attach that dataset in the next session and copy both directories back into `/kaggle/working` before running another phase. Recovery files contain optimizer, scaler, RNG, and position state; inference files are lightweight model or adapter states.